# 5.2 Career matching vs. O\*NET's official Interest Profiler

The app re-implements O\*NET's published matching rule (Gregory & Lewis 2016, pp. 11–12; ROADMAP 2.6): Pearson r between the user's six RIASEC scores and each occupation's six interest scores; **Best fit** r ≥ .729, **Great fit** r ≥ .608, and **Good fit** (0 ≤ r < .608) only to fill the list up to 10 per Job Zone.

Here it is compared with O\*NET's own implementation, the `/mnm/interestprofiler/careers` endpoint of O\*NET Web Services, on **60 random profiles (seed 2026) × the 4 Job Zones**, plus one flat profile.

- Both sides use O\*NET 31.0 (the local database since CHANGELOG 024).
- The app's scores are checkbox counts (0–10 per area); the API takes 0–40, so they are sent ×4. Pearson r doesn't change with scale.
- API answers are cached in `data/analysis/onet_ip_careers_cache.json` (untracked), so a rerun calls the API only for profiles it hasn't seen. The key is read from `.env` (`ONET_API_KEY`) and never printed.

In [1]:
import _setup
_setup.setup()

In [2]:
import json
import os
import time
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

from career_rag.interest_profiler_local import RIASEC_INTERESTS
from career_rag.ip_career_matcher import (BEST_FIT_MIN_R, GREAT_FIT_MIN_R, ONET_JOB_ZONES, load_zone_occupations,
                                          onet_fit_list, pearson)

CACHE = Path("data/analysis/onet_ip_careers_cache.json")
rng = np.random.default_rng(2026)
profiles = [dict(zip(RIASEC_INTERESTS, map(int, rng.integers(0, 11, 6)))) for _ in range(60)]
flat = dict.fromkeys(RIASEC_INTERESTS, 5)
print(f"{len(profiles)} random profiles, e.g. {profiles[0]}")

60 random profiles, e.g. {'Realistic': 9, 'Investigative': 1, 'Artistic': 0, 'Social': 7, 'Enterprising': 4, 'Conventional': 5}


In [3]:
load_dotenv(".env")
cache = json.loads(CACHE.read_text(encoding="utf-8")) if CACHE.exists() else {}


def official(scores, zone):
    # O*NET's list for these scores and Job Zone: [(code, title, fit)], or the HTTP error it returned.
    key = ",".join(str(scores[i]) for i in RIASEC_INTERESTS) + f"|{zone}"
    if key not in cache:
        params = {name.lower(): scores[name] * 4 for name in RIASEC_INTERESTS} | {"zone": zone, "start": 1, "end": 1000}
        request = urllib.request.Request(
            "https://api-v2.onetcenter.org/mnm/interestprofiler/careers?" + urllib.parse.urlencode(params),
            headers={"X-API-Key": os.environ["ONET_API_KEY"], "Accept": "application/json"})
        try:
            with urllib.request.urlopen(request, timeout=30) as response:
                body = json.loads(response.read().decode("utf-8"))
            cache[key] = [[c["code"], c["title"], c.get("fit", "")] for c in body.get("career", [])]
        except urllib.error.HTTPError as error:
            cache[key] = {"http_error": error.code}
        time.sleep(0.2)
    return cache[key]


answers = {(p, zone): official(profile, zone) for p, profile in enumerate(profiles) for zone in ONET_JOB_ZONES}
flat_answers = {zone: official(flat, zone) for zone in ONET_JOB_ZONES}
CACHE.parent.mkdir(parents=True, exist_ok=True)
CACHE.write_text(json.dumps(cache, indent=1, sort_keys=True), encoding="utf-8")
print(f"O*NET lists: {len(answers)}; errors: {sum(isinstance(a, dict) for a in answers.values())}")

O*NET lists: 240; errors: 0


## 1. Same careers, same labels?

In [4]:
FIT = {"Best fit": "Best", "Great fit": "Great", "Good fit": "Good"}
zone_occupations = {zone: load_zone_occupations(zone) for zone in ONET_JOB_ZONES}
user_vectors = {}
rows, differences = [], []
for (p, zone), theirs in answers.items():
    ours = {m["onet_soc_code"]: FIT[m["fit"]] for m in onet_fit_list(profiles[p], zone_occupations[zone])}
    theirs = {code: fit for code, _, fit in theirs}
    common = ours.keys() & theirs.keys()
    rows.append({"profile": p, "zone": zone, "ours": len(ours), "onet": len(theirs), "common": len(common),
                 "same_list": ours.keys() == theirs.keys(), "same_labels": sum(ours[c] == theirs[c] for c in common)})
    for code in ours.keys() | theirs.keys():
        if ours.get(code) != theirs.get(code):
            occupation = next(o for o in zone_occupations[zone] if o["onet_soc_code"] == code)
            r = pearson([float(profiles[p][i]) for i in RIASEC_INTERESTS], occupation["interest_scores"])
            differences.append({"profile": p, "zone": zone, "career": occupation["career_title"], "r": r,
                                "ours": ours.get(code, "not listed"), "onet": theirs.get(code, "not listed")})
lists = pd.DataFrame(rows)
differences = pd.DataFrame(differences)
total = lists.ours.sum() + lists.onet.sum() - lists.common.sum()
print(f"lists compared: {len(lists)}; identical career sets: {lists.same_list.sum()} ({lists.same_list.mean():.1%})")
print(f"careers listed by either: {total:,}; by both: {lists.common.sum():,} ({lists.common.sum() / total:.2%})")
print(f"same fit label, among careers listed by both: {lists.same_labels.sum():,} of {lists.common.sum():,} "
      f"({lists.same_labels.sum() / lists.common.sum():.2%})")
print(f"list length: ours {lists.ours.min()}-{lists.ours.max()}, O*NET {lists.onet.min()}-{lists.onet.max()}")

lists compared: 240; identical career sets: 176 (73.3%)
careers listed by either: 7,183; by both: 7,117 (99.08%)
same fit label, among careers listed by both: 7,107 of 7,117 (99.86%)
list length: ours 6-238, O*NET 6-238


## 2. The differences, by kind

Each career where the two sides disagree is one of three kinds:
- **label differs:** both list it, with a different fit;
- **fill stops earlier:** we list it as the 10th, lowest Good fit, and O\*NET's list stops at 9;
- **listed by one side only** for another reason.

In [5]:
lengths = {key: len(answer) for key, answer in answers.items()}


def kind(row):
    if row.ours != "not listed" and row.onet != "not listed":
        return "label differs"
    if row.ours == "Good" and row.onet == "not listed" and lengths[(row.profile, row.zone)] == 9:
        return "fill stops earlier"
    return "listed by one side only"


differences["kind"] = differences.apply(kind, axis=1)
print(differences.kind.value_counts().to_string())
filled = [len(a) for a in answers.values() if any(fit == "Good" for _, _, fit in a)]
print(f"O*NET lists that add Good fits: {len(filled)}; their length: "
      + ", ".join(f"{n}: {filled.count(n)}" for n in sorted(set(filled), reverse=True)))

kind
fill stops earlier         53
listed by one side only    13
label differs              10
O*NET lists that add Good fits: 119; their length: 10: 67, 9: 50, 7: 1, 6: 1


**Labels and one-side listings.** These sit on a cut-off (.729 or .608), or are near-ties for the last place of a filled list, and they go both ways: O\*NET says Best at r = 0.7288 and Great at r = 0.7291. So it isn't O\*NET rounding r (tested below); the likely cause is that O\*NET computes r from interest scores with more decimals than the two it publishes, which moves a few careers across a line.

In [6]:
differences[differences.kind != "fill stops earlier"].sort_values("r").round({"r": 4}).reset_index(drop=True)

,profile,zone,career,r,ours,onet,kind
0,40,4,Recreation Workers,0.2046,not listed,Good,listed by one side only
1,40,4,Broadcast Announcers and Radio Disc Jockeys,0.2052,Good,not listed,listed by one side only
2,52,3,Forensic Science Technicians,0.4729,not listed,Good,listed by one side only
3,52,3,Radiation Therapists,0.4736,Good,not listed,listed by one side only
4,47,5,Urologists,0.6079,not listed,Great,listed by one side only
5,12,4,Proofreaders and Copy Markers,0.6079,not listed,Great,listed by one side only
6,52,5,"Architecture Teachers, Postsecondary",0.6079,not listed,Great,listed by one side only
7,6,4,"Farmers, Ranchers, and Other Agricultural Mana...",0.6079,not listed,Great,listed by one side only
8,39,3,First-Line Supervisors of Production and Opera...,0.6080,not listed,Great,listed by one side only
9,55,5,"Education Administrators, Kindergarten through...",0.6082,Great,Good,label differs


In [7]:
def fit_rounded(r, digits):
    r = round(r, digits)
    return "Best" if r >= BEST_FIT_MIN_R else "Great" if r >= GREAT_FIT_MIN_R else "Good"


labelled = differences[differences.kind == "label differs"]
for digits in (2, 3, 4):
    fixed = sum(fit_rounded(row.r, digits) == row.onet for row in labelled.itertuples())
    print(f"rounding r to {digits} decimals reproduces O*NET's label for {fixed} of {len(labelled)}")

rounding r to 2 decimals reproduces O*NET's label for 7 of 10
rounding r to 3 decimals reproduces O*NET's label for 4 of 10
rounding r to 4 decimals reproduces O*NET's label for 0 of 10


**The fill.** The published rule fills the list up to 10 with Good fits. O\*NET's API does that in some lists and stops at 9 in others, and the API's answer alone doesn't show why (O\*NET may count an occupation it doesn't display). The careers we add are the next-best match, so this changes the end of a list, not its top.

## 3. A flat profile (all six scores equal)

In [8]:
print("O*NET:", {zone: (a if isinstance(a, dict) else dict(pd.Series([f for _, _, f in a]).value_counts()))
                 for zone, a in flat_answers.items()})
print("the app:", {zone: len(onet_fit_list(flat, zone_occupations[zone])) for zone in ONET_JOB_ZONES},
      "careers: r is undefined for a flat profile, so the app shows a warning instead (CHANGELOG 019)")

O*NET: {2: {'Good': np.int64(337)}, 3: {'Good': np.int64(208)}, 4: {'Good': np.int64(226)}, 5: {'Good': np.int64(152)}}
the app: {2: 0, 3: 0, 4: 0, 5: 0} careers: r is undefined for a flat profile, so the app shows a warning instead (CHANGELOG 019)


## Findings

The numbers above are recomputed on every run (from the cache, once filled).

1. **The app's matching is O\*NET's matching.** Across 240 profile × Job Zone lists, about 99% of the careers either side lists are listed by both, and among those, fit labels agree in more than 99.8%. This is the first evaluation number for the career test.
2. **Fit labels differ only at the cut-offs**, in both directions, which points to O\*NET using more precise interest scores than it publishes. Rounding r doesn't reproduce O\*NET's labels, so there's no simple fix, and none is needed: a career at r = 0.729 is a borderline Best/Great either way.
3. **The end of the list differs more often than the rule predicts.** In about 2 of 5 lists that add Good fits, O\*NET stops at 9 where its published rule (and the app) fills to 10. That is why only about 3 in 4 lists are *identical* although almost every career agrees. Students see the extra career last, labelled Good fit.
4. **Flat profiles:** O\*NET lists every career in the zone as a Good fit; the app shows a warning instead, as the matching paper's rule 9 asks. That is a deliberate difference.

**What this does not test:** whether the *profile* is right (the 60-item short form is O\*NET's own instrument, scored the same way), or whether the careers suit Turkish students (Phase 6). It also uses random profiles; real users' profiles are less uniform, with one or two clear high points.